### **State Reducers**

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph

#### **Default overwriting state**

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    graph_state: int

def node_1(state):
    print('---Node 1---')
    return {'graph_state': state['graph_state'] + 1}

# Build graph
builder = StateGraph(State)
builder.add_node('node_1', node_1)

# Logic
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

# Invocation
graph.invoke({'graph_state' : 1})

### **Branching**

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    graph_state: int

def node_1(state):
    print('---Node 1---')
    return {'graph_state': state['graph_state'] + 1}

def node_2(state):
    print('---Node 2---')
    return {'graph_state': state['graph_state'] + 1}

def node_3(state):
    print('---Node 3---')
    return {'graph_state': state['graph_state'] + 1}

# Build graph
builder = StateGraph(State)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

# Logic
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_1', 'node_3')
builder.add_edge('node_2', END)
builder.add_edge('node_3', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({'graph_state' : 1})
except InvalidUpdateError as e:
    print(f'InvalidUpdateError occurred: {e}')

We see a problem!

Node 1 branches to nodes 2 and 3. Nodes 2 and 3 run in parallel, which means they run in the same step of the graph. They both attempt to overwrite the state within the same step. This is ambiguous for the graph! Which state should it keep?

#### **Reducers**

In [ ]:
from operator import add
from typing import Annotated

class DefaultReducerState(TypedDict):
    graph_state: Annotated[list[int], add]

def node_1(state):
    print('---Node 1---')
    return {'graph_state': [state['graph_state'][-1] + 1]}

def node_2(state):
    print('---Node 2---')
    return {'graph_state': [state['graph_state'][-1] + 1]}

def node_3(state):
    print('---Node 3---')
    return {'graph_state': [state['graph_state'][-1] + 1]}

# Build graph
builder = StateGraph(DefaultReducerState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

# Logic
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_1', 'node_3')
builder.add_edge('node_2', END)
builder.add_edge('node_3', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

# Invocation
graph.invoke({'graph_state': [1]})

In [ ]:
try:
    graph.invoke({'graph_state' : None})
except TypeError as e:
    print(f'TypeError occurred: {e}')

We see an error because `operator.add` attempts to concatenate `NoneType` pass as input to list in `node_1`.

#### **Custom Reducers**

In [20]:
def reduce_list(left: list | None, right: list | None) -> list:
    '''Safely combine two lists, handling cases where either or both inputs might be None.

    Args:
        left (list | None): The first list to combine, or None.
        right (list | None): The second list to combine, or None.

    Returns:
        list: A new list containing all elements from both input lists.
               If an input is None, it's treated as an empty list.
    '''
    if not left:
        left = []
    if not right:
        right = []
    return left + right

class CustomReducerState(TypedDict):
    graph_state: Annotated[list[int], reduce_list]

In [ ]:
def node_1(state):
    print('---Node 1---')
    return {'graph_state': [2]}

# Build graph
builder = StateGraph(CustomReducerState)
builder.add_node('node_1', node_1)

# Logic
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({'graph_state' : None}))
except TypeError as e:
    print(f'TypeError occurred: {e}')

#### **Messages**
2 ways to define `MessagesState`
- Manually code
- Use the built-in shortcut

In [22]:
from typing import Annotated
from langgraph.graph import MessagesState
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

from langchain_core.messages import AIMessage, HumanMessage

# Define a custom TypedDict that includes a list of messages with add_messages reducer
class CustomMessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    added_key_1: str
    added_key_2: str
    # etc

# Use MessagesState, which includes the messages key with add_messages reducer
class ExtendedMessagesState(MessagesState):
    # Add any keys needed beyond messages, which is pre-built 
    added_key_1: str
    added_key_2: str
    # etc

In [ ]:
# Initial state
initial_messages = [
    AIMessage(content = 'Hello! How can I assist you?', name = 'Model', id='0'),
    HumanMessage(content ='I\'m looking for information on marine biology.', name = 'Lance', id = '1.1')
]

# New message to add
new_message = HumanMessage(content = 'I\'m looking for information on whales, specifically', name = 'Lance', id = '1.1')

# Test - Overwritten message with the same ID
add_messages(initial_messages , new_message)

In [ ]:
from langchain_core.messages import RemoveMessage
import pprint

# Message list
messages = [AIMessage(content = 'Hi.', name = 'Bot', id = '1')]
messages.append(HumanMessage(content = 'Hi.', name = 'Lance', id = '2'))
messages.append(AIMessage(content = 'So you said you were researching ocean mammals?', name = 'Bot', id = '3'))
messages.append(HumanMessage(content = 'Yes, I know about whales. But what others should I learn about?', name = 'Lance', id = '1.1'))

# Isolate messages to delete
delete_messages = [RemoveMessage(id = m.id) for m in messages[:-2]]
pprint.pprint(messages)